# 11 — Feature Engineering and Preprocessing

## 1. Objective and scope boundary

This notebook freezes the feature policy only. It decides which features may proceed into feature-engineering experiments, without creating those features or performing preprocessing.

```text
Notebook 10: What does the data tell us?
        ↓
Notebook 11 policy freeze: Which features are allowed to proceed?
        ↓
Next work: How are approved features created?
```

> The policy freeze establishes feature eligibility before transformation. This prevents preprocessing code from implicitly deciding which columns are allowed into the model.

An approved candidate is allowed into deterministic feature-creation design; it is **not** guaranteed to be a final model feature.

In [1]:
from hashlib import sha256
import json
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from urban_ops.features.policy import (
    PolicyStatus,
    feature_policy_table,
    load_feature_policy,
    validate_feature_policy_evidence,
)

POLICY_PATH = PROJECT_ROOT / 'configs/features/resolution_risk_baseline.yaml'

def file_state(path):
    return (sha256(path.read_bytes()).hexdigest(), path.stat().st_mtime_ns)

latest_pointer = PROJECT_ROOT / 'data/splits/resolution_risk/latest.json'
latest_payload = json.loads(latest_pointer.read_text(encoding='utf-8'))
configured_run = Path(latest_payload['run_path'])
split_run = configured_run if configured_run.is_absolute() else PROJECT_ROOT / configured_run
governed_inputs = [
    POLICY_PATH,
    latest_pointer,
    split_run / 'train.parquet',
    split_run / 'validation.parquet',
    split_run / 'test.parquet',
    split_run / 'split_metadata.json',
    split_run / 'split_rules_snapshot.yaml',
]
governed_inputs.extend(sorted((PROJECT_ROOT / 'reports/11_split_aware_eda').rglob('*')))
governed_inputs = tuple(path for path in governed_inputs if path.is_file())
before_states = {str(path.relative_to(PROJECT_ROOT)): file_state(path) for path in governed_inputs}

pd.Series({'governed_input_count': len(before_states), 'split_id': latest_payload['split_id']})

governed_input_count                                   55
split_id                20260806T135114Z_9d945cb2da0eecfc
dtype: object

## 2. Notebook 10 handoff authority

Notebook 10 completed its analysis and reconciliation, but reported `model_ready: False`. That is correct: EDA describes the data and recommends possibilities, but it does not freeze which representations are permitted to proceed.

The authority order is: Step 4 leakage and prediction-time availability; Notebook 10 leakage audit; Notebook 10's newer `eda_status`; train-only structure evidence; then the older `baseline_decision`. A statistically interesting feature can remain excluded or conditional because governance takes priority over descriptive EDA patterns.

In [2]:
policy = load_feature_policy(POLICY_PATH)
policy_table = feature_policy_table(policy)
display(pd.Series(policy.notebook_10_handoff, name='Notebook 10 handoff'))
display(pd.DataFrame(policy.authority_hierarchy).sort_values('priority'))

split_id            20260806T135114Z_9d945cb2da0eecfc
integrity                                        PASS
reconciliation                                   PASS
step_9a_decision                             COMPLETE
model_ready                                     False
Name: Notebook 10 handoff, dtype: object

,priority,authority,source
0,1,Step 4 leakage and prediction-time availability,docs/leakage_policy.md
1,2,Notebook 10 leakage audit,reports/11_split_aware_eda/tables/leakage_audi...
2,3,Notebook 10 EDA statuses,notebooks/10_split_aware_eda.ipynb
3,4,"Notebook 10 missingness, cardinality, and outl...",reports/11_split_aware_eda/
4,5,Notebook 10 baseline recommendations,reports/11_split_aware_eda/tables/baseline_fea...


## 3. Feature-policy decision rules

A feature receives permission to proceed only after all of these gates pass:

1. available at the complaint-creation prediction moment;
2. safe under Step 4 leakage governance;
3. non-null with usable train variation;
4. the approved representation rather than an alternative or redundant form; and
5. no unresolved policy or temporal-generalization blocker.

Target-rate differences alone can never grant feature-creation eligibility.

## 4. Approved first-pass candidates

The four preferred temporal representations are frozen as candidates. Their common source is retained, but no calendar field is derived during this work.

In [3]:
approved = policy_table.loc[policy_table['policy_status'].eq('APPROVED_CANDIDATE')]
display(approved[['feature_name', 'source_column', 'policy_status', 'eda_status', 'reason', 'phase_2_allowed']])
assert tuple(approved['feature_name']) == policy.feature_creation_allow_list

,feature_name,source_column,policy_status,eda_status,reason,phase_2_allowed
1,created_hour,created_date,APPROVED_CANDIDATE,CANDIDATE,Preferred creation-time-safe representation fo...,True
2,created_day_of_week,created_date,APPROVED_CANDIDATE,CANDIDATE,Preferred machine-friendly creation-time weekd...,True
3,created_month,created_date,APPROVED_CANDIDATE,CANDIDATE,Preferred interpretable creation-time seasonal...,True
4,is_weekend,created_date,APPROVED_CANDIDATE,CANDIDATE,Simple creation-time-safe operational grouping...,True


## 5. Alternative and redundant representations

Names and numeric calendar codes carry the same essential information for day and month, so the first baseline selects one representation. Quarter and week-of-year overlap with the preferred month representation and remain under redundancy review.

In [4]:
representation_review = policy_table.loc[policy_table['policy_status'].isin([
    'ALTERNATIVE_REPRESENTATION', 'REVIEW_REDUNDANCY', 'REVIEW'
])]
display(representation_review[['feature_name', 'policy_status', 'preferred_counterpart', 'redundancy_status', 'reason', 'phase_2_allowed']])

,feature_name,policy_status,preferred_counterpart,redundancy_status,reason,phase_2_allowed
5,created_day_name,ALTERNATIVE_REPRESENTATION,created_day_of_week,EXACT_ALTERNATIVE,Human-readable duplicate representation of the...,False
6,created_month_name,ALTERNATIVE_REPRESENTATION,created_month,EXACT_ALTERNATIVE,Human-readable duplicate representation of the...,False
7,created_quarter,REVIEW_REDUNDANCY,created_month,OVERLAPS_PREFERRED,Coarser seasonal representation that overlaps ...,False
8,created_week_of_year,REVIEW_REDUNDANCY,created_month,OVERLAPS_PREFERRED,Seasonal representation that overlaps strongly...,False
9,created_day_of_month,REVIEW,NaN,NONE,"Creation-time safe, but current EDA does not j...",False


## 6. Conditional features

`created_year` may encode time progression or regime rather than a durable operational pattern. Geography remains unresolved because the current authority does not prove creation-time availability and immutability. Missingness strategies and apparent target associations do not resolve that uncertainty.

In [5]:
conditional = policy_table.loc[policy_table['policy_status'].eq('CONDITIONAL')]
display(conditional[['feature_name', 'prediction_time_status', 'leakage_status', 'reason', 'phase_2_allowed']])
assert not conditional['phase_2_allowed'].any()

,feature_name,prediction_time_status,leakage_status,reason,phase_2_allowed
10,created_year,AVAILABLE,SAFE,May encode temporal progression or regime rath...,False
11,borough,UNRESOLVED,CONDITIONAL,Prediction-time availability and later mutabil...,False
12,location_type,UNRESOLVED,CONDITIONAL,Prediction-time availability and later mutabil...,False
13,incident_zip,UNRESOLVED,CONDITIONAL,Prediction-time availability and later mutabil...,False
14,latitude,UNRESOLVED,CONDITIONAL,Prediction-time availability and later mutabil...,False
15,longitude,UNRESOLVED,CONDITIONAL,Prediction-time availability and later mutabil...,False


## 7. Excluded features

Exclusions retain explicit reasons: leakage/post-creation/target-derived, identifier, all-null, or zero variance. `unique_key` remains available only for traceability. `due_date` remains blocked because its prediction-time availability and mutability are unproven.

In [6]:
excluded = policy_table.loc[policy_table['policy_status'].str.startswith('EXCLUDE_')]
display(excluded[['feature_name', 'policy_status', 'prediction_time_status', 'leakage_status', 'variation_status', 'reason', 'phase_2_allowed']])
assert not excluded['phase_2_allowed'].any()

,feature_name,policy_status,prediction_time_status,leakage_status,variation_status,reason,phase_2_allowed
16,unique_key,EXCLUDE_IDENTIFIER,AVAILABLE,BLOCKED,HAS_VARIATION,Identifier is retained for traceability only a...,False
17,descriptor_2,EXCLUDE_ALL_NULL,UNRESOLVED,CONDITIONAL,ALL_NULL,Training evidence identifies the field as enti...,False
18,agency,EXCLUDE_ZERO_VARIANCE,AVAILABLE,SAFE,ZERO_VARIANCE,Fixed by the selected scope and has one traini...,False
19,agency_name,EXCLUDE_ZERO_VARIANCE,AVAILABLE,SAFE,ZERO_VARIANCE,Fixed by the selected scope and has one traini...,False
20,complaint_type,EXCLUDE_ZERO_VARIANCE,AVAILABLE,SAFE,ZERO_VARIANCE,Fixed by the selected scope and has one traini...,False
21,descriptor,EXCLUDE_ZERO_VARIANCE,UNRESOLVED,CONDITIONAL,ZERO_VARIANCE,Training evidence shows no useful variation; e...,False
22,open_data_channel_type,EXCLUDE_ZERO_VARIANCE,AVAILABLE,SAFE,ZERO_VARIANCE,Training evidence shows one fixed channel valu...,False
23,closed_date,EXCLUDE_LEAKAGE,UNAVAILABLE,BLOCKED,HAS_VARIATION,Closure outcome is unavailable at complaint cr...,False
24,due_date,EXCLUDE_LEAKAGE,UNRESOLVED,BLOCKED,HAS_VARIATION,Target input whose creation-time timing and mu...,False
25,status,EXCLUDE_LEAKAGE,UNAVAILABLE,BLOCKED,ZERO_VARIANCE,Final mutable outcome status is post-creation ...,False


## 8. Frozen feature policy

This is the complete required policy table. The YAML remains the machine-readable source of truth; this view is generated from the validated loader.

In [7]:
required_columns = [
    'feature_name', 'source_column', 'policy_status', 'prediction_time_status',
    'leakage_status', 'eda_status', 'redundancy_status', 'reason', 'phase_2_allowed'
]
display(policy_table[required_columns])
assert policy_table['feature_name'].is_unique
assert policy_table[required_columns].notna().all().all()

,feature_name,source_column,policy_status,prediction_time_status,leakage_status,eda_status,redundancy_status,reason,phase_2_allowed
0,created_date,created_date,SOURCE_ONLY,AVAILABLE,SAFE,REVIEW,SOURCE_FOR_DERIVATIONS,Required to derive temporal features; the raw ...,False
1,created_hour,created_date,APPROVED_CANDIDATE,AVAILABLE,SAFE,CANDIDATE,NONE,Preferred creation-time-safe representation fo...,True
2,created_day_of_week,created_date,APPROVED_CANDIDATE,AVAILABLE,SAFE,CANDIDATE,NONE,Preferred machine-friendly creation-time weekd...,True
3,created_month,created_date,APPROVED_CANDIDATE,AVAILABLE,SAFE,CANDIDATE,NONE,Preferred interpretable creation-time seasonal...,True
4,is_weekend,created_date,APPROVED_CANDIDATE,AVAILABLE,SAFE,CANDIDATE,NONE,Simple creation-time-safe operational grouping...,True
5,created_day_name,created_date,ALTERNATIVE_REPRESENTATION,AVAILABLE,SAFE,ALTERNATIVE_REPRESENTATION,EXACT_ALTERNATIVE,Human-readable duplicate representation of the...,False
6,created_month_name,created_date,ALTERNATIVE_REPRESENTATION,AVAILABLE,SAFE,ALTERNATIVE_REPRESENTATION,EXACT_ALTERNATIVE,Human-readable duplicate representation of the...,False
7,created_quarter,created_date,REVIEW_REDUNDANCY,AVAILABLE,SAFE,REVIEW_REDUNDANCY,OVERLAPS_PREFERRED,Coarser seasonal representation that overlaps ...,False
8,created_week_of_year,created_date,REVIEW_REDUNDANCY,AVAILABLE,SAFE,REVIEW_REDUNDANCY,OVERLAPS_PREFERRED,Seasonal representation that overlaps strongly...,False
9,created_day_of_month,created_date,REVIEW,AVAILABLE,SAFE,REVIEW,NONE,"Creation-time safe, but current EDA does not j...",False


## 9. Policy validation

The validator reconciles the policy against Notebook 10's complete baseline inventory, Step 4/Notebook 10 leakage audit, all-null evidence, and zero-variance evidence. The final check also confirms every governed split artifact and Notebook 10 report has the same hash and modification time as at notebook start.

In [8]:
validate_feature_policy_evidence(policy)
after_states = {str(path.relative_to(PROJECT_ROOT)): file_state(path) for path in governed_inputs}
boundary_flags = [
    'transformations_implemented', 'missing_values_imputed',
    'rare_categories_grouped', 'encoder_fitted', 'scaler_fitted',
    'column_transformer_built', 'feature_matrix_created', 'model_trained'
]
policy_checks = pd.Series({
    'notebook_10_complete': policy.notebook_10_handoff['step_9a_decision'] == 'COMPLETE',
    'notebook_10_not_model_ready': policy.notebook_10_handoff['model_ready'] is False,
    'exact_feature_creation_allow_list': policy.feature_creation_allow_list == ('created_hour', 'created_day_of_week', 'created_month', 'is_weekend'),
    'only_approved_candidates_allowed': policy_table.loc[policy_table['phase_2_allowed'], 'policy_status'].eq('APPROVED_CANDIDATE').all(),
    'implementation_scope_respected': all(policy.implementation_boundary[flag] is False for flag in boundary_flags),
    'governed_sources_unchanged': before_states == after_states,
}, name='passed')
display(policy_checks)
assert policy_checks.all()

notebook_10_complete                 True
notebook_10_not_model_ready          True
exact_feature_creation_allow_list    True
only_approved_candidates_allowed     True
implementation_scope_respected       True
governed_sources_unchanged           True
Name: passed, dtype: bool

## 10. Completion decision

The policy freeze is complete when the policy and evidence reconcile, only approved candidates are allowed to proceed, and governed inputs remain unchanged. The frozen decision is now consumed by deterministic feature creation below.

In [9]:
pd.Series({
    'policy_decision': 'COMPLETE' if policy_checks.all() else 'NOT_COMPLETE',
    'model_ready': False,
    'feature_creation_allow_list': ', '.join(policy.feature_creation_allow_list),
    'policy_handoff': 'READY_FOR_DETERMINISTIC_CREATION',
})

policy_decision                                                         COMPLETE
model_ready                                                                False
feature_creation_allow_list    created_hour, created_day_of_week, created_mon...
policy_handoff                                  READY_FOR_DETERMINISTIC_CREATION
dtype: object

## Deterministic Feature Creation

The frozen policy decided **which** derived features are allowed. This section implements **how** those approved features are calculated.

Deterministic creation learns no statistics from training data, so it requires no fitting. The same timestamp always produces the same values, independent of row order, targets, validation/test behavior, randomness, locale, current time, or external services.

Missing-value handling, category grouping, encoding, scaling, outlier transformation, persisted model matrices, and model training remain intentionally deferred.

### A. Frozen policy handoff

The builder consumes the already validated policy. It does not maintain a second allow-list or reinterpret EDA evidence. `created_date` remains a source-only dependency rather than a direct baseline feature.

In [10]:
from urban_ops.eda.pipeline import load_eda_config
from urban_ops.eda.source import load_verified_split, resolve_split_run
from urban_ops.features.temporal import (
    approved_temporal_feature_names,
    build_feature_reconciliation_table,
    build_temporal_validation_table,
    derive_split_temporal_features,
)

EDA_CONFIG_PATH = PROJECT_ROOT / 'configs/eda/resolution_risk.yaml'
eda_config = load_eda_config(EDA_CONFIG_PATH)
split_run_path = resolve_split_run(
    split_root=eda_config.split_root,
    latest_pointer=eda_config.latest_pointer,
)
split_source = load_verified_split(
    run_path=split_run_path,
    latest_pointer=eda_config.latest_pointer,
    required_completion_status=eda_config.required_completion_status,
    identifier_column=eda_config.identifier_column,
    target_column=eda_config.target_column,
    timestamp_column=eda_config.timestamp_column,
)
source_frames = {
    'train': split_source.train,
    'validation': split_source.validation,
    'test': split_source.test,
}
source_snapshots = {name: frame.copy(deep=True) for name, frame in source_frames.items()}
pd.Series({name: len(frame) for name, frame in source_frames.items()}, name='source_rows')

train         23699
validation     6762
test           5499
Name: source_rows, dtype: int64

### B. Approved derivation allow-list

A feature is derived only when the frozen policy marks it `APPROVED_CANDIDATE` and allows it to proceed. Conditional geography and non-preferred calendar representations remain outside the created baseline set.

In [11]:
approved_names = approved_temporal_feature_names(policy)
approved_policy_rows = policy_table.loc[policy_table['feature_name'].isin(approved_names)]
display(approved_policy_rows[['feature_name', 'source_column', 'policy_status', 'phase_2_allowed', 'reason']])
assert approved_names == policy.feature_creation_allow_list
assert policy.by_name['created_date'].policy_status is PolicyStatus.SOURCE_ONLY

,feature_name,source_column,policy_status,phase_2_allowed,reason
1,created_hour,created_date,APPROVED_CANDIDATE,True,Preferred creation-time-safe representation fo...
2,created_day_of_week,created_date,APPROVED_CANDIDATE,True,Preferred machine-friendly creation-time weekd...
3,created_month,created_date,APPROVED_CANDIDATE,True,Preferred interpretable creation-time seasonal...
4,is_weekend,created_date,APPROVED_CANDIDATE,True,Simple creation-time-safe operational grouping...


### C. Source-column contract

Published split timestamps must already be non-null, timezone-aware UTC datetimes. The builder does not parse strings, coerce malformed values, infer a timezone, or use the local machine timezone. Calendar components therefore retain the authoritative UTC interpretation used by cleaning and splitting.

In [12]:
source_contract = pd.DataFrame([
    {
        'split_name': name,
        'row_count': len(frame),
        'created_date_dtype': str(frame['created_date'].dtype),
        'timezone': str(frame['created_date'].dt.tz),
        'null_count': int(frame['created_date'].isna().sum()),
        'contract_valid': str(frame['created_date'].dt.tz) == 'UTC' and not frame['created_date'].isna().any(),
    }
    for name, frame in source_frames.items()
])
display(source_contract)
assert source_contract['contract_valid'].all()

,split_name,row_count,created_date_dtype,timezone,null_count,contract_valid
0,train,23699,"datetime64[us, UTC]",UTC,0,True
1,validation,6762,"datetime64[us, UTC]",UTC,0,True
2,test,5499,"datetime64[us, UTC]",UTC,0,True


### D. Temporal derivation rules

All definitions are pure calendar extraction from `created_date`: hour uses 0–23; weekday uses Pandas' Monday=0 through Sunday=6 convention; month uses 1–12; weekend is true exactly for weekday values 5 and 6.

In [13]:
derivation_rules = pd.DataFrame([
    {'feature_name': 'created_hour', 'source': 'created_date', 'rule': 'UTC hour component', 'dtype': 'Int8', 'domain': '0..23'},
    {'feature_name': 'created_day_of_week', 'source': 'created_date', 'rule': 'Monday=0 through Sunday=6', 'dtype': 'Int8', 'domain': '0..6'},
    {'feature_name': 'created_month', 'source': 'created_date', 'rule': 'calendar month number', 'dtype': 'Int8', 'domain': '1..12'},
    {'feature_name': 'is_weekend', 'source': 'created_date', 'rule': 'created_day_of_week in {5, 6}', 'dtype': 'bool', 'domain': '{False, True}'},
])
display(derivation_rules)

,feature_name,source,rule,dtype,domain
0,created_hour,created_date,UTC hour component,Int8,0..23
1,created_day_of_week,created_date,Monday=0 through Sunday=6,Int8,0..6
2,created_month,created_date,calendar month number,Int8,1..12
3,is_weekend,created_date,"created_day_of_week in {5, 6}",bool,"{False, True}"


### E. Training derivation preview

This small preview is calculated from actual training timestamps. It is evidence of the reusable builder's output, not hardcoded example data.

In [14]:
derived_frames = derive_split_temporal_features(source_frames, policy=policy)
preview_columns = ['created_date', *approved_names]
display(derived_frames['train'][preview_columns].head(8))

,created_date,created_hour,created_day_of_week,created_month,is_weekend
0,2024-01-01 07:58:15+00:00,7,0,1,False
1,2024-01-01 08:00:34+00:00,8,0,1,False
2,2024-01-01 08:01:53+00:00,8,0,1,False
3,2024-01-01 08:03:15+00:00,8,0,1,False
4,2024-01-01 08:03:59+00:00,8,0,1,False
5,2024-01-01 08:04:35+00:00,8,0,1,False
6,2024-01-01 08:05:18+00:00,8,0,1,False
7,2024-01-01 08:06:21+00:00,8,0,1,False


### F. Split consistency and feature-domain validation

The identical builder is applied independently to train, validation, and test. Later splits validate domains only; they do not select features or alter derivation rules. Repeatability compares values exactly across two independent calls.

In [15]:
feature_validation = build_temporal_validation_table(
    source_frames, derived_frames, policy=policy
)
display(feature_validation)
assert feature_validation['creation_status'].eq('COMPLETE').all()
assert feature_validation['domain_valid'].all()
assert feature_validation['deterministic'].all()
weekend_check = feature_validation.loc[
    feature_validation['feature_name'].eq('is_weekend'),
    'weekend_relationship_valid',
]
assert weekend_check.eq(True).all()

,feature_name,source_column,policy_status,created,dtype,non_null_count,unique_count,minimum,maximum,domain_valid,train_valid,validation_valid,test_valid,weekend_relationship_valid,deterministic,creation_status
0,created_hour,created_date,APPROVED_CANDIDATE,True,Int8,23699,24,0,23,True,True,True,True,None,True,COMPLETE
1,created_day_of_week,created_date,APPROVED_CANDIDATE,True,Int8,23699,7,0,6,True,True,True,True,None,True,COMPLETE
2,created_month,created_date,APPROVED_CANDIDATE,True,Int8,23699,12,1,12,True,True,True,True,None,True,COMPLETE
3,is_weekend,created_date,APPROVED_CANDIDATE,True,bool,23699,2,False,True,True,True,True,True,True,True,COMPLETE


### G. Row and target reconciliation

Feature creation may add approved columns only. Row count, index, order, complaint identifier, target, and every original source value must reconcile exactly.

In [16]:
feature_reconciliation = build_feature_reconciliation_table(
    source_frames,
    derived_frames,
    identifier_column=eda_config.identifier_column,
    target_column=eda_config.target_column,
)
display(feature_reconciliation)
reconciliation_columns = [
    'row_count_preserved', 'index_preserved', 'row_order_preserved',
    'unique_key_preserved', 'target_preserved', 'source_values_preserved',
]
assert feature_reconciliation[reconciliation_columns].all().all()

,split_name,rows_before,rows_after,row_count_preserved,index_preserved,row_order_preserved,unique_key_preserved,target_preserved,source_values_preserved
0,train,23699,23699,True,True,True,True,True,True
1,validation,6762,6762,True,True,True,True,True,True
2,test,5499,5499,True,True,True,True,True,True


### H. Source immutability and implementation boundary

All derived frames exist in memory only. Source DataFrames, split artifacts, Notebook 10 reports, and the frozen policy must remain byte-for-byte and modification-time unchanged. Non-approved temporal fields are not created, and conditional geography is not promoted or transformed.

In [17]:
repeat_frames = derive_split_temporal_features(source_frames, policy=policy)
creation_after_states = {str(path.relative_to(PROJECT_ROOT)): file_state(path) for path in governed_inputs}
source_frames_unchanged = all(
    source_frames[name].equals(source_snapshots[name]) for name in source_frames
)
repeat_identical = all(
    derived_frames[name].equals(repeat_frames[name]) for name in derived_frames
)
new_columns = {
    name: tuple(column for column in derived_frames[name] if column not in source_frames[name])
    for name in source_frames
}
only_approved_created = all(columns == approved_names for columns in new_columns.values())
creation_boundary_checks = pd.Series({
    'source_frames_unchanged': source_frames_unchanged,
    'repeat_derivation_identical': repeat_identical,
    'only_approved_features_created': only_approved_created,
    'governed_files_unchanged': before_states == creation_after_states,
    'no_rows_added_or_removed': feature_reconciliation['row_count_preserved'].all(),
    'no_preprocessing_or_modelling': all(policy.implementation_boundary[flag] is False for flag in boundary_flags),
}, name='passed')
display(creation_boundary_checks)
assert creation_boundary_checks.all()

source_frames_unchanged           True
repeat_derivation_identical       True
only_approved_features_created    True
governed_files_unchanged          True
no_rows_added_or_removed          True
no_preprocessing_or_modelling     True
Name: passed, dtype: bool

### I. Completion decision

Deterministic feature creation is complete when policy authority, source contracts, exact repeatability, domains, reconciliation, and immutability all pass. The result remains an in-memory analytical view, not a persisted model matrix.

**STOP:** the next reviewed work is **Handle Categorical Missing Values**. It is intentionally not implemented here.

In [18]:
creation_complete = (
    feature_validation['creation_status'].eq('COMPLETE').all()
    and feature_reconciliation[reconciliation_columns].all().all()
    and creation_boundary_checks.all()
)
pd.Series({
    'creation_decision': 'COMPLETE' if creation_complete else 'NOT_COMPLETE',
    'model_ready': False,
    'created_features': ', '.join(approved_names),
    'next_work': 'Handle Categorical Missing Values',
})

creation_decision                                             COMPLETE
model_ready                                                      False
created_features     created_hour, created_day_of_week, created_mon...
next_work                            Handle Categorical Missing Values
dtype: object